# Table 2

Audio-visual video classification results on VGGSounder. We report multi-label classification metrics (subset accuracy, F1- score, Hit accuracy, modality confusion μ) on background music free subset for audio- a(A), visual - v(V), audio-visual - av(AV), audio-only - a(A¬V) and video-only - v(V¬A) inputs. The embedding models CAV-MAE, DeepAVFusion, and Equi-AV were finetuned on the VGGSound training set. We report metrics for k = 1 here and for other k in Appendix D. The closed sourced multi-modal foundation models Gemini and open-sourced models use a zero-shot evaluation protocol and LLM-assisted protocol respectively.

In [1]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

vggsounder = VGGSounder()
main_table = benchmark(
    models_path="../../supplimentary/models-results",
    display_names = {
        "cav-mae": "CAV-MAE",
        "deepavfusion": "DeepAVFusion",
        "equiav": "Equi-AV",
        "avsiam": "AV-Siam",
        "gemini-1.5-flash": "Gemini 1.5 Flash",
        "gemini-1.5-pro": "Gemini 1.5 Pro",
        "gemini-2.0-flash": "Gemini 2.0 Flash",
        "video-llama-2-av": "VideoLLaMA 2",
        "unified-io-2": "Unified-IO 2",
        "pandagpt": "PandaGPT",
        "ola": "OLA",
    },
    metrics=[
        ("accuracy", ["a", "v", "av"]), 
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    vggsounder=vggsounder
)

  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


In [2]:
import sklearn; print(sklearn.__file__)

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/__init__.py


In [3]:
print(main_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 17.94 & 24.21 & 29.61 & 39.53 & 38.87 & 47.24 & 21.69 & 23.56 & 67.07 & 56.37 & 67.51 & 4.07 & 4.93 & 0.85 \\
DeepAVFusion & 13.89 & 13.83 & 26.16 & 29.06 & 23.29 & 41.81 & 15.67 & 13.65 & 49.40 & 33.80 & 59.79 & 4.21 & 3.42 & 0.26 \\
Equi-AV & 15.80 & 13.57 & 24.40 & 33.65 & 22.93 & 39.33 & 19.02 & 14.39 & 57.09 & 33.25 & 56.21 & 7.77 & 6.28 & 1.51 \\
AV-Siam & 17.40 & 24.99 & 27.86 & 38.12 & 39.77 & 43.73 & 20.41 & 23.77 & 64.69 & 57.68 & 62.49 & 10.76 & 8.55 & 3.98 \\
Gemini 1.5 Flash & 2.36 & 18.55 & 20.24 & 14.21 & 42.02 & 46.82 & 17.06 & 30.96 & 30.75 & 51.79 & 62.30 & 13.25 & 4.12 & 1.45 \\
Gemini 1.5 Pro & 3.48 & 27.18 & 26.91 & 19.38 & 54.90 & 57.20 & 20.02 & 32.14 & 33.43 & 73.27 & 78.36 & 2.65 & 4.03 & 0.52 \\
Gemi

# Table 3
Difference between labels that contain confounding factors such as "background music" and that do not contain.

In [4]:
from vggsounder.benchmark import compute_all_results, get_metric_dataframe, select_metrics
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
vggsounder = VGGSounder(csv_path=csv_path)

# background_music
vggsounder.set_meta_filters(background_music=True)
background_music_ids = [video.video_id for video in vggsounder]

background_music_only_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=background_music_ids,
    dataset_path=csv_path
)

# no background music
vggsounder.set_meta_filters(background_music=False)
no_background_music_ids = [video.video_id for video in vggsounder]

no_background_music_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=no_background_music_ids,
    dataset_path=csv_path
)

# voice over
vggsounder.set_meta_filters(voice_over=True)
voice_over_ids = [video.video_id for video in vggsounder]

voice_over_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=voice_over_ids,
    dataset_path=csv_path
)

# no voice over
vggsounder.set_meta_filters(voice_over=False)
no_voice_over_ids = [video.video_id for video in vggsounder]

no_voice_over_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=no_voice_over_ids,
    dataset_path=csv_path
)

# static image
vggsounder.set_meta_filters(static_image=True)
static_image_ids = [video.video_id for video in vggsounder]

static_image_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=static_image_ids,
    dataset_path=csv_path
)

# no static image
vggsounder.set_meta_filters(static_image=False)
no_static_image_ids = [video.video_id for video in vggsounder]

no_static_image_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=no_static_image_ids,
    dataset_path=csv_path
)

# neither
vggsounder.set_meta_filters(
    static_image=False, 
    voice_over=False, 
    background_music=False
)
neither_ids = [video.video_id for video in vggsounder]

neither_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=neither_ids,
    dataset_path=csv_path
)

# main table -- vggsounder + background music
either_of_confounder = list(set(background_music_ids + voice_over_ids + static_image_ids))
main_table_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=either_of_confounder,
    dataset_path=csv_path
)

  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

...

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


In [5]:
main_table_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    dataset_path=csv_path
)

  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


In [6]:
# background music only
background_music_only_df = get_metric_dataframe(background_music_only_results)
background_music_only_table = select_metrics(background_music_only_df, [("f1", ["a", "v", "av"])], display_names=display_names)

# no background music
no_background_music_df = get_metric_dataframe(no_background_music_results)
no_background_music_table = select_metrics(no_background_music_df, [("f1", ["a", "v", "av"])], display_names=display_names)

# voice over only
voice_over_df = get_metric_dataframe(voice_over_results)
voice_over_table = select_metrics(voice_over_df, [("f1", ["a"])], display_names=display_names)

# no voice over
no_voice_over_df = get_metric_dataframe(no_voice_over_results)
no_voice_over_table = select_metrics(no_voice_over_df, [("f1", ["a"])], display_names=display_names)

# static image only
static_image_df = get_metric_dataframe(static_image_results)
static_image_table = select_metrics(static_image_df, [("f1", ["a", "v"])], display_names=display_names)
static_image_accuracy_table = select_metrics(static_image_df, [("accuracy", ["a", "v"])], display_names=display_names)

# no static image
no_static_image_df = get_metric_dataframe(no_static_image_results)
no_static_image_table = select_metrics(no_static_image_df, [("f1", ["a", "v"])], display_names=display_names)
no_static_image_accuracy_table = select_metrics(no_static_image_df, [("accuracy", ["a", "v"])], display_names=display_names)

# neither
neither_df = get_metric_dataframe(neither_results)
neither_table = select_metrics(neither_df, [("f1", ["a", "v", "av"])], display_names=display_names)

# main table -- vggsounder + background music
main_table_df = get_metric_dataframe(main_table_results)
main_table_table = select_metrics(main_table_df, [("f1", ["a", "v", "av"])], display_names=display_names)

In [7]:
import pandas as pd

# final diff table 
diff_tables = [
    background_music_only_table - no_background_music_table,
    voice_over_table - no_voice_over_table,
    static_image_table - no_static_image_table,
    static_image_accuracy_table,
    no_static_image_accuracy_table,
    main_table_table - neither_table,
]

diff_tables = pd.concat(diff_tables, axis=1)
diff_tables

metric                  f1                                                     \
modality                 a         v        av          a         a         v   
CAV-MAE          -6.800304 -2.269950 -3.453754 -11.927253  4.934742 -5.037180   
DeepAVFusion     -7.380979 -3.644781 -5.443862 -12.093060  3.290829 -5.445197   
Equi-AV          -7.578338 -2.852391 -5.114559 -10.716678  5.119869 -7.423582   
AV-Siam          -7.158799 -4.599701 -6.001623 -12.460195  4.549937 -9.689779   
Gemini 1.5 Flash  1.377227 -3.453764 -4.351181  23.729605 -4.471900 -2.175175   
Gemini 1.5 Pro    1.500017 -0.853707 -0.815063  23.733890 -4.476055 -0.745318   
Gemini 2.0 Flash -0.783692 -2.703378 -4.194068  -1.876072  1.346644 -2.665566   
VideoLLaMA 2     -4.165877 -2.344206 -2.038672  -7.622562  4.610964 -3.015794   
Unified-IO 2     -9.881900  3.117212 -2.421686  -3.453694  1.633012 -3.409818   
PandaGPT         -4.126162  1.177635 -0.305579   7.269517 -6.331558 -3.022335   
OLA              -8.579443  3.099229  2.594631  15.492189 -6.841991 -7.343210   

metric             accuracy                                         f1  \
modality                  a          v          a          v         a   
CAV-MAE           25.310174  26.747720  15.683777  24.091257 -3.423096   
DeepAVFusion      18.427835  15.396825  12.065200  13.460177 -3.976091   
Equi-AV           22.456576  11.246201  13.638067  13.569422 -3.407652   
AV-Siam           24.503722  23.708207  15.121768  24.704180 -3.704658   
Gemini 1.5 Flash   2.109181  19.604863   2.435369  18.396186  4.773532   
Gemini 1.5 Pro     3.349876  32.826748   3.634320  27.019665  5.411947   
Gemini 2.0 Flash   4.342432  18.693009   1.925815  15.876394 -0.570177   
VideoLLaMA 2      21.774194  25.987842  14.694642  23.657104 -2.040328   
Unified-IO 2      20.285360  14.893617  13.293368  15.016600 -2.252492   
PandaGPT           2.295285   6.990881   3.334582   5.465225  0.580047   
OLA               16.873449   8.662614  14.799550  10.947476  1.958281   

metric                                
modality                 v        av  
CAV-MAE          -1.198819 -1.281174  
DeepAVFusion     -0.695457 -2.362134  
Equi-AV          -1.080755 -2.244214  
AV-Siam          -1.706783 -1.870224  
Gemini 1.5 Flash -1.348141 -1.672929  
Gemini 1.5 Pro   -0.972645 -1.737121  
Gemini 2.0 Flash -0.842923 -1.337647  
VideoLLaMA 2     -0.543463 -0.446518  
Unified-IO 2      0.437680 -0.902420  
PandaGPT          0.251575 -0.059304  
OLA               0.041444  0.116915

In [8]:
print(diff_tables.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{6}{r}{f1} & \multicolumn{4}{r}{accuracy} & \multicolumn{3}{r}{f1} \\
modality & a & v & av & a & a & v & a & v & a & v & a & v & av \\
\midrule
CAV-MAE & -6.80 & -2.27 & -3.45 & -11.93 & 4.93 & -5.04 & 25.31 & 26.75 & 15.68 & 24.09 & -3.42 & -1.20 & -1.28 \\
DeepAVFusion & -7.38 & -3.64 & -5.44 & -12.09 & 3.29 & -5.45 & 18.43 & 15.40 & 12.07 & 13.46 & -3.98 & -0.70 & -2.36 \\
Equi-AV & -7.58 & -2.85 & -5.11 & -10.72 & 5.12 & -7.42 & 22.46 & 11.25 & 13.64 & 13.57 & -3.41 & -1.08 & -2.24 \\
AV-Siam & -7.16 & -4.60 & -6.00 & -12.46 & 4.55 & -9.69 & 24.50 & 23.71 & 15.12 & 24.70 & -3.70 & -1.71 & -1.87 \\
Gemini 1.5 Flash & 1.38 & -3.45 & -4.35 & 23.73 & -4.47 & -2.18 & 2.11 & 19.60 & 2.44 & 18.40 & 4.77 & -1.35 & -1.67 \\
Gemini 1.5 Pro & 1.50 & -0.85 & -0.82 & 23.73 & -4.48 & -0.75 & 3.35 & 32.83 & 3.63 & 27.02 & 5.41 & -0.97 & -1.74 \\
Gemini 2.0 Flash & -0.78 & -2.70 & -4.19 & -1.88 & 1.35 & -2.67 & 4.34 & 18.69 & 1.93 & 15

# Figure 9 (Table)
Performance of state-of-the-art models on VGGSound. 

In [9]:
import pickle as pk
from vggsounder.benchmark import benchmark, get_metric_dataframe, select_metrics
from vggsounder.labels import VGGSounder


models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
vggsounder = VGGSounder(csv_path=csv_path)

vggsound_results = pk.load(open("precomputed_results/vggsound_results.pkl", "rb"))
table_9_df = get_metric_dataframe(vggsound_results, modalities=["a", "v", "av"])
table_9 = select_metrics(
    table_9_df, 
    [
        ("accuracy", ["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ], 
    display_names=display_names
)
table_9
    

metric             accuracy                               mu            \
modality                  a          v         av          a         v   
CAV-MAE           59.050887  45.565195  65.078337   4.713194  4.836204   
DeepAVFusion      40.824576  27.243331  53.098895   4.176772  3.172999   
Equi-AV           46.678752  24.841383  50.077690   6.907937  5.509517   
AV-Siam           56.907937  47.267901  55.250550  13.174932  8.921404   
Gemini 1.5 Flash   0.310760  22.122232  23.598343   1.514955  4.169364   
Gemini 1.5 Pro     1.288359  25.767189  21.306487   1.618542  5.412405   
Gemini 2.0 Flash   5.703742  20.290043  19.390133   2.499029  4.771462   
VideoLLaMA 2      27.981354  17.007640  21.461867  11.161466  2.848634   
Unified-IO 2      32.280202  20.238249  52.395442   4.875049  3.418361   
PandaGPT           5.198757   7.652467   8.947300   4.506021  4.480124   
OLA               10.714748   8.630066  14.294963   7.613622  4.052829   

metric                      
modality                av  
CAV-MAE           0.666839  
DeepAVFusion      0.067367  
Equi-AV           0.977599  
AV-Siam           3.923346  
Gemini 1.5 Flash  0.090638  
Gemini 1.5 Pro    0.239544  
Gemini 2.0 Flash  0.627994  
VideoLLaMA 2      1.417843  
Unified-IO 2      0.874013  
PandaGPT          0.938754  
OLA               0.712158

In [10]:
print(table_9.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av \\
\midrule
CAV-MAE & 59.05 & 45.57 & 65.08 & 4.71 & 4.84 & 0.67 \\
DeepAVFusion & 40.82 & 27.24 & 53.10 & 4.18 & 3.17 & 0.07 \\
Equi-AV & 46.68 & 24.84 & 50.08 & 6.91 & 5.51 & 0.98 \\
AV-Siam & 56.91 & 47.27 & 55.25 & 13.17 & 8.92 & 3.92 \\
Gemini 1.5 Flash & 0.31 & 22.12 & 23.60 & 1.51 & 4.17 & 0.09 \\
Gemini 1.5 Pro & 1.29 & 25.77 & 21.31 & 1.62 & 5.41 & 0.24 \\
Gemini 2.0 Flash & 5.70 & 20.29 & 19.39 & 2.50 & 4.77 & 0.63 \\
VideoLLaMA 2 & 27.98 & 17.01 & 21.46 & 11.16 & 2.85 & 1.42 \\
Unified-IO 2 & 32.28 & 20.24 & 52.40 & 4.88 & 3.42 & 0.87 \\
PandaGPT & 5.20 & 7.65 & 8.95 & 4.51 & 4.48 & 0.94 \\
OLA & 10.71 & 8.63 & 14.29 & 7.61 & 4.05 & 0.71 \\
\bottomrule
\end{tabular}



# Table 7
**Audio-visual video classification results on VGGSounder for k ∈ {3, 5, 10}.** The table is vertically grouped by k. Within each block, the four models are compared across the three metrics and input modalities

In [11]:
from vggsounder.benchmark import benchmark
import pandas as pd

vggsounder = VGGSounder()
# top 3 table
top_3_table = benchmark(
    models_path="../../supplimentary/models-results",
    display_names = {
        "cav-mae": "CAV-MAE",
        "deepavfusion": "DeepAVFusion",
        "equiav": "Equi-AV",
        "avsiam": "AV-Siam",
    },
    metrics=[
        ("accuracy@3", ["a", "v", "av"]),
        ("f1@3", ["a", "v", "av"]),
        ("hit_rate@3", ["a", "v", "av"]),
    ],
    vggsounder=vggsounder
)

# top 5 table
top_5_table = benchmark(
    models_path="../../supplimentary/models-results",
    display_names = {
        "cav-mae": "CAV-MAE",
        "deepavfusion": "DeepAVFusion",
        "equiav": "Equi-AV",
        "avsiam": "AV-Siam",
    },
    metrics=[
        ("accuracy@5", ["a", "v", "av"]),
        ("f1@5", ["a", "v", "av"]),
        ("hit_rate@5", ["a", "v", "av"]),
    ],
    vggsounder=vggsounder
)

# top 10 table
top_10_table = benchmark(
    models_path="../../supplimentary/models-results",
    display_names = {
        "cav-mae": "CAV-MAE",
        "deepavfusion": "DeepAVFusion",
        "equiav": "Equi-AV",
        "avsiam": "AV-Siam",
    },
    metrics=[
        ("accuracy@10", ["a", "v", "av"]),
        ("f1@10", ["a", "v", "av"]),
        ("hit_rate@10", ["a", "v", "av"]),
    ],
    vggsounder=vggsounder
)

# rename columns 
new_cols = [(col[0].split("@")[0], col[1]) for col in top_10_table.columns.to_list()]
top_3_table.columns = pd.MultiIndex.from_tuples(new_cols)
top_5_table.columns = pd.MultiIndex.from_tuples(new_cols)
top_10_table.columns = pd.MultiIndex.from_tuples(new_cols)

# concat to get the final table
table_7 = pd.concat([top_3_table, top_5_table, top_10_table])
table_7

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

accuracy                             f1                        \
                     a         v        av          a          v         av   
CAV-MAE       1.225141  0.706714  1.405602  41.962133  37.929653  45.345933   
DeepAVFusion  0.306614  0.051125  0.852632  30.486420  23.848849  39.779077   
Equi-AV       0.738441  0.235571  0.586510  36.249358  24.108645  36.574806   
AV-Siam       1.174792  0.873577  1.041561  40.148109  39.311754  43.073016   
CAV-MAE       0.025174  0.009815  0.010112  36.315557  32.093883  37.005691   
DeepAVFusion  0.000000  0.000000  0.010526  26.453874  20.390449  32.462790   
Equi-AV       0.008391  0.000000  0.000000  31.721351  21.180654  30.010911   
AV-Siam       0.008391  0.019631  0.020224  34.622238  32.992888  35.537141   
CAV-MAE       0.000000  0.000000  0.000000  25.674358  22.004289  24.349561   
DeepAVFusion  0.000000  0.000000  0.000000  18.992645  14.584407  21.478654   
Equi-AV       0.000000  0.000000  0.000000  22.905157  15.433850  20.249516   
AV-Siam       0.000000  0.000000  0.000000  24.417723  22.464533  23.902718   

               hit_rate                        
                      a          v         av  
CAV-MAE       83.183687  74.420887  84.154111  
DeepAVFusion  67.884363  52.678937  77.094737  
Equi-AV       76.336326  49.656459  73.445242  
AV-Siam       81.295628  75.500589  81.666498  
CAV-MAE       87.949987  80.477032  88.563050  
DeepAVFusion  74.577311  60.490798  82.421053  
Equi-AV       82.655031  57.126031  79.128324  
AV-Siam       86.061928  81.203377  86.712509  
CAV-MAE       92.162457  86.690224  92.739407  
DeepAVFusion  82.163820  69.642127  87.410526  
Equi-AV       88.730385  66.666667  85.327131  
AV-Siam       90.702358  87.387122  92.011326

In [12]:
print(table_7.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrr}
\toprule
 & \multicolumn{3}{r}{accuracy} & \multicolumn{3}{r}{f1} & \multicolumn{3}{r}{hit_rate} \\
 & a & v & av & a & v & av & a & v & av \\
\midrule
CAV-MAE & 1.23 & 0.71 & 1.41 & 41.96 & 37.93 & 45.35 & 83.18 & 74.42 & 84.15 \\
DeepAVFusion & 0.31 & 0.05 & 0.85 & 30.49 & 23.85 & 39.78 & 67.88 & 52.68 & 77.09 \\
Equi-AV & 0.74 & 0.24 & 0.59 & 36.25 & 24.11 & 36.57 & 76.34 & 49.66 & 73.45 \\
AV-Siam & 1.17 & 0.87 & 1.04 & 40.15 & 39.31 & 43.07 & 81.30 & 75.50 & 81.67 \\
CAV-MAE & 0.03 & 0.01 & 0.01 & 36.32 & 32.09 & 37.01 & 87.95 & 80.48 & 88.56 \\
DeepAVFusion & 0.00 & 0.00 & 0.01 & 26.45 & 20.39 & 32.46 & 74.58 & 60.49 & 82.42 \\
Equi-AV & 0.01 & 0.00 & 0.00 & 31.72 & 21.18 & 30.01 & 82.66 & 57.13 & 79.13 \\
AV-Siam & 0.01 & 0.02 & 0.02 & 34.62 & 32.99 & 35.54 & 86.06 & 81.20 & 86.71 \\
CAV-MAE & 0.00 & 0.00 & 0.00 & 25.67 & 22.00 & 24.35 & 92.16 & 86.69 & 92.74 \\
DeepAVFusion & 0.00 & 0.00 & 0.00 & 18.99 & 14.58 & 21.48 & 82.16 & 69.64 & 87.41 \\
Equi-

# Table 8 
**Audio-visual video classification results on the subset of VGGSounder that is labelled as containing background music**

In [13]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
vggsounder = VGGSounder(csv_path=csv_path)

# background_music
vggsounder.set_meta_filters(background_music=True)
background_music_ids = [video.video_id for video in vggsounder]

background_music_only_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=background_music_ids,
    dataset_path=csv_path
)
background_music_only_table

  0%|          | 0/11 [00:00<?, ?it/s]

metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE           11.940789  24.312134  28.212423  32.726932  36.602394   
DeepAVFusion       8.347415  12.325581  24.070450  21.683035  19.643469   
Equi-AV            9.835526  12.855210  22.285443  26.069017  20.079787   
AV-Siam           11.151316  23.094272  24.087245  30.963991  35.172872   
Gemini 1.5 Flash   2.565789  18.042400  17.354196  15.583193  38.565318   
Gemini 1.5 Pro     4.078947  28.010825  27.121859  20.875421  54.044750   
Gemini 2.0 Flash   1.217105  14.704556  12.043623  12.080037  35.969739   
VideoLLaMA 2      12.368421  23.319802  26.979611  36.599087  47.938343   
Unified-IO 2       9.243421  16.644114  27.643433  27.808327  33.628319   
PandaGPT           2.532895   6.450158   7.633950  14.455206  20.563771   
OLA               11.217105  12.133514  22.522523  39.892127  29.616648   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           43.783209  20.336788  31.275720  57.401316  49.661705   
DeepAVFusion      36.366959  13.978138  18.143460  38.053396  26.651163   
Equi-AV           34.218916  16.483161  15.226337  45.723684  27.244023   
AV-Siam           37.725824  19.170984  35.802469  54.309211  47.722147   
Gemini 1.5 Flash  42.467043  13.117641  33.891993  34.506579  47.135769   
Gemini 1.5 Pro    56.382678  16.467872  37.919463  38.750000  71.673433   
Gemini 2.0 Flash  36.306051  10.650510  32.179226  19.243421  41.632837   
VideoLLaMA 2      51.626984  24.597919  44.665012  51.809211  47.722147   
Unified-IO 2      47.681576  20.239789  28.923077  39.342105  31.303563   
PandaGPT          21.244160   9.893993  19.571865  16.743421  16.914750   
OLA               48.636189  31.718694  27.167630  48.519737  26.792963   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           58.605974   5.270049  4.582651  0.589198  
DeepAVFusion      48.679061   3.295225  2.824479  0.100874  
Equi-AV           45.803698   7.790507  5.106383  1.014730  
AV-Siam           50.497866  13.878887  7.037643  3.142390  
Gemini 1.5 Flash  56.661925  17.250409  3.338789  1.178396  
Gemini 1.5 Pro    77.572309   5.630115  3.666121  0.949264  
Gemini 2.0 Flash  45.851114   3.404255  3.698854  0.981997  
VideoLLaMA 2      52.489331  20.621931  4.779051  2.782324  
Unified-IO 2      53.437648   9.623568  4.680851  1.407529  
PandaGPT          16.737790   9.656301  6.121113  2.716858  
OLA               50.023708  14.173486  6.350245  2.193126

In [14]:
print(background_music_only_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 11.94 & 24.31 & 28.21 & 32.73 & 36.60 & 43.78 & 20.34 & 31.28 & 57.40 & 49.66 & 58.61 & 5.27 & 4.58 & 0.59 \\
DeepAVFusion & 8.35 & 12.33 & 24.07 & 21.68 & 19.64 & 36.37 & 13.98 & 18.14 & 38.05 & 26.65 & 48.68 & 3.30 & 2.82 & 0.10 \\
Equi-AV & 9.84 & 12.86 & 22.29 & 26.07 & 20.08 & 34.22 & 16.48 & 15.23 & 45.72 & 27.24 & 45.80 & 7.79 & 5.11 & 1.01 \\
AV-Siam & 11.15 & 23.09 & 24.09 & 30.96 & 35.17 & 37.73 & 19.17 & 35.80 & 54.31 & 47.72 & 50.50 & 13.88 & 7.04 & 3.14 \\
Gemini 1.5 Flash & 2.57 & 18.04 & 17.35 & 15.58 & 38.57 & 42.47 & 13.12 & 33.89 & 34.51 & 47.14 & 56.66 & 17.25 & 3.34 & 1.18 \\
Gemini 1.5 Pro & 4.08 & 28.01 & 27.12 & 20.88 & 54.04 & 56.38 & 16.47 & 37.92 & 38.75 & 71.67 & 77.57 & 5.63 & 3.67 & 0.95 \\
Gemini

# Table 9 
**Audio-visual video classification results on the subset of VGGSounder that is labelled as not containing background music**

In [15]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
vggsounder = VGGSounder(csv_path=csv_path)


# no background music
vggsounder.set_meta_filters(background_music=False)
no_background_music_ids = [video.video_id for video in vggsounder]

no_background_music_only_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=no_background_music_ids,
    dataset_path=csv_path
)
no_background_music_only_table

  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE           17.940757  24.214762  29.608656  39.527236  38.872343   
DeepAVFusion      13.893999  13.834356  26.157895  29.064014  23.288250   
Equi-AV           15.800957  13.574794  24.400849  33.647356  22.932178   
AV-Siam           17.403709  24.990185  27.859238  38.122790  39.772573   
Gemini 1.5 Flash   2.357976  18.551237  20.244716  14.205966  42.019081   
Gemini 1.5 Pro     3.482420  27.179034  26.908686  19.375403  54.898458   
Gemini 2.0 Flash   2.433498  16.313310  15.481849  12.863730  38.673117   
VideoLLaMA 2      16.245699  23.881037  27.697442  40.764964  50.282548   
Unified-IO 2      15.272300  14.654495  29.709779  37.690227  30.511106   
PandaGPT           3.398506   5.349431   6.299929  18.581368  19.386136   
OLA               15.993958  10.541814  19.000910  48.471570  26.517419   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           47.236963  21.690243  23.555071  67.072250  56.370239   
DeepAVFusion      41.810821  15.672160  13.651877  49.399912  33.803681   
Equi-AV           39.333475  19.018257  14.394766  57.094906  33.254810   
AV-Siam           43.727446  20.408657  23.773173  64.689100  57.675697   
Gemini 1.5 Flash  46.818224  17.058824  30.961924  30.754384  51.786415   
Gemini 1.5 Pro    57.197741  20.015488  32.142857  33.431233  73.272477   
Gemini 2.0 Flash  40.500119   7.728232  26.452282  18.863808  47.006282   
VideoLLaMA 2      53.665656  25.551896  38.012015  58.152220  53.219474   
Unified-IO 2      50.103262  27.433628  21.296296  54.426450  32.175108   
PandaGPT          21.549739  17.655786  18.730650  19.719728  17.098547   
OLA               46.041558  45.534743  17.352941  57.497692  24.842952   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           67.509354   4.070251  4.927584  0.849009  
DeepAVFusion      59.789474   4.213727  3.423110  0.260643  
Equi-AV           56.213975   7.774263  6.284335  1.506576  
AV-Siam           62.493680  10.762444  8.548360  3.978692  
Gemini 1.5 Flash  62.301547  13.251207  4.120193  1.448310  
Gemini 1.5 Pro    78.359794   2.646912  4.028633  0.524388  
Gemini 2.0 Flash  50.561230   2.588647  4.786083  1.023806  
VideoLLaMA 2      59.915057  14.508074  5.427002  3.113035  
Unified-IO 2      63.363333  10.795738  4.636258  1.839521  
PandaGPT          18.737992   9.447311  6.334277  3.029799  
OLA               50.197189  17.338106  6.234393  2.846679

In [16]:
print(no_background_music_only_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 17.94 & 24.21 & 29.61 & 39.53 & 38.87 & 47.24 & 21.69 & 23.56 & 67.07 & 56.37 & 67.51 & 4.07 & 4.93 & 0.85 \\
DeepAVFusion & 13.89 & 13.83 & 26.16 & 29.06 & 23.29 & 41.81 & 15.67 & 13.65 & 49.40 & 33.80 & 59.79 & 4.21 & 3.42 & 0.26 \\
Equi-AV & 15.80 & 13.57 & 24.40 & 33.65 & 22.93 & 39.33 & 19.02 & 14.39 & 57.09 & 33.25 & 56.21 & 7.77 & 6.28 & 1.51 \\
AV-Siam & 17.40 & 24.99 & 27.86 & 38.12 & 39.77 & 43.73 & 20.41 & 23.77 & 64.69 & 57.68 & 62.49 & 10.76 & 8.55 & 3.98 \\
Gemini 1.5 Flash & 2.36 & 18.55 & 20.24 & 14.21 & 42.02 & 46.82 & 17.06 & 30.96 & 30.75 & 51.79 & 62.30 & 13.25 & 4.12 & 1.45 \\
Gemini 1.5 Pro & 3.48 & 27.18 & 26.91 & 19.38 & 54.90 & 57.20 & 20.02 & 32.14 & 33.43 & 73.27 & 78.36 & 2.65 & 4.03 & 0.52 \\
Gemi

# Table 10 
**Audio-visual video classification results on the subset of VGGSounder that is labelled as containing static images**

In [17]:
from vggsounder.benchmark import benchmark, compute_all_results
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}

# static image
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(static_image=True)
static_image_ids = [video.video_id for video in vggsounder]

static_image_results = compute_all_results(
    models_path=models_path,
    models_filter=display_names.keys(),
    subset_ids=static_image_ids,
    dataset_path=csv_path
)

static_image_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=static_image_ids,
    dataset_path=csv_path
)
static_image_table

  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE           25.310174  26.747720  33.762058  42.556570  33.681623   
DeepAVFusion      18.427835  15.396825  28.260870  30.471484  17.469493   
Equi-AV           22.456576  11.246201  26.205788  36.681223  15.365704   
AV-Siam           24.503722  23.708207  30.064309  40.730449  29.748002   
Gemini 1.5 Flash   2.109181  19.604863  22.990354  10.468394  39.348534   
Gemini 1.5 Pro     3.349876  32.826748  32.636656  15.642644  54.038680   
Gemini 2.0 Flash   4.342432  18.693009  17.202572  13.907419  35.672131   
VideoLLaMA 2      21.774194  25.987842  33.601286  44.068561  47.014925   
Unified-IO 2      20.285360  14.893617  33.601286  37.184030  27.763496   
PandaGPT           2.295285   6.990881   8.360129  11.979308  16.681943   
OLA               16.873449   8.662614  22.829582  40.524983  19.981998   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           42.084433  39.937598  24.113475  66.501241  41.641337   
DeepAVFusion      35.463918  28.756058  10.606061  47.680412  21.587302   
Equi-AV           33.905013  34.217369   8.510638  57.320099  18.996960   
AV-Siam           38.918206  37.545502  22.695035  63.647643  36.778116   
Gemini 1.5 Flash  43.586550   9.785408  28.965517  22.022333  43.009119   
Gemini 1.5 Pro    56.494611  14.492129  36.257310  25.434243  64.741641   
Gemini 2.0 Flash  38.009050  13.340364  23.188406  19.292804  38.905775   
VideoLLaMA 2      52.875280  40.512362  35.658915  57.258065  42.097264   
Unified-IO 2      48.923077  35.516373  17.582418  47.208437  22.948328   
PandaGPT          20.306513  10.919751  11.235955  12.282878  12.310030   
OLA               43.524904  36.854888  14.736842  44.727047  15.045593   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           51.286174   5.672010  3.267571  0.493218  
DeepAVFusion      43.143813   4.481434  2.752881  0.256082  
Equi-AV           41.318328   9.001233  3.575832  0.678175  
AV-Siam           47.427653  16.646116  5.425401  2.836005  
Gemini 1.5 Flash  50.964630  12.515413  3.822441  0.739827  
Gemini 1.5 Pro    70.900322   4.993835  4.007398  0.863132  
Gemini 2.0 Flash  43.408360   3.267571  3.205919  0.924784  
VideoLLaMA 2      51.125402  23.119605  3.514180  2.466091  
Unified-IO 2      50.160772   9.062885  3.082614  1.233046  
PandaGPT          15.273312   7.706535  4.562269  1.726264  
OLA               42.122186  17.447596  3.390875  1.171393

In [18]:
print(static_image_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 25.31 & 26.75 & 33.76 & 42.56 & 33.68 & 42.08 & 39.94 & 24.11 & 66.50 & 41.64 & 51.29 & 5.67 & 3.27 & 0.49 \\
DeepAVFusion & 18.43 & 15.40 & 28.26 & 30.47 & 17.47 & 35.46 & 28.76 & 10.61 & 47.68 & 21.59 & 43.14 & 4.48 & 2.75 & 0.26 \\
Equi-AV & 22.46 & 11.25 & 26.21 & 36.68 & 15.37 & 33.91 & 34.22 & 8.51 & 57.32 & 19.00 & 41.32 & 9.00 & 3.58 & 0.68 \\
AV-Siam & 24.50 & 23.71 & 30.06 & 40.73 & 29.75 & 38.92 & 37.55 & 22.70 & 63.65 & 36.78 & 47.43 & 16.65 & 5.43 & 2.84 \\
Gemini 1.5 Flash & 2.11 & 19.60 & 22.99 & 10.47 & 39.35 & 43.59 & 9.79 & 28.97 & 22.02 & 43.01 & 50.96 & 12.52 & 3.82 & 0.74 \\
Gemini 1.5 Pro & 3.35 & 32.83 & 32.64 & 15.64 & 54.04 & 56.49 & 14.49 & 36.26 & 25.43 & 64.74 & 70.90 & 4.99 & 4.01 & 0.86 \\
Gemini

# Table 11 
**Audio-visual video classification results on the subset of VGGSounder that is labelled as not containing static images**

In [19]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}

# no static image
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(static_image=False)
no_static_image_ids = [video.video_id for video in vggsounder]

no_static_image_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=no_static_image_ids,
    dataset_path=csv_path
)
no_static_image_table


  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE           15.683777  24.091257  29.122714  37.621828  38.718802   
DeepAVFusion      12.065200  13.460177  25.653207  27.180655  22.914690   
Equi-AV           13.638067  13.569422  23.909986  31.561354  22.789286   
AV-Siam           15.121768  24.704180  27.039381  36.180512  39.437782   
Gemini 1.5 Flash   2.435369  18.396186  19.558720  14.940294  41.523709   
Gemini 1.5 Pro     3.634320  27.019665  26.635021  20.118699  54.783998   
Gemini 2.0 Flash   1.925815  15.876394  14.750352  12.560774  38.337697   
VideoLLaMA 2      14.694642  23.657104  27.241561  39.457598  50.030719   
Unified-IO 2      13.293368  15.016600  29.113924  35.551018  31.173314   
PandaGPT           3.334582   5.465225   6.434599  18.310867  19.704278   
OLA               14.799550  10.947476  19.444444  47.366974  27.325208   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           46.876158  17.528614  25.240936  64.938179  55.929173   
DeepAVFusion      41.151762  12.433236  14.857143  46.989549  33.123894   
Equi-AV           38.696135  15.091140  14.960991  54.477332  32.919043   
AV-Siam           42.906532  16.511234  26.525929  62.450356  56.967736   
Gemini 1.5 Flash  46.204926  17.286698  31.742044  32.663919  51.400358   
Gemini 1.5 Pro    57.085262  19.933899  33.149577  35.608842  73.448540   
Gemini 2.0 Flash  39.869842   7.564881  27.882508  18.898464  46.445901   
VideoLLaMA 2      53.362698  22.139902  39.429929  56.815287  52.804971   
Unified-IO 2      49.752027  23.501384  23.137255  51.862121  32.527454   
PandaGPT          21.552008  16.452516  19.346405  19.940052  17.332085   
OLA               46.587547  42.963356  19.615146  56.995129  25.759768   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           66.745781   4.149624  5.049453  0.832900  
DeepAVFusion      58.624155   3.969974  3.366352  0.224423  
Equi-AV           55.098453   7.629955  6.343422  1.494757  
AV-Siam           61.093530  10.760764  8.581840  3.926526  
Gemini 1.5 Flash  61.875879  14.248531  3.978583  1.472447  
Gemini 1.5 Pro    78.621660   3.041571  3.948836  0.580055  
Gemini 2.0 Flash  50.079114   2.692050  4.729679  1.026251  
VideoLLaMA 2      59.018987  14.858333  5.510523  3.115937  
Unified-IO 2      62.245077  10.738455  4.833792  1.814531  
PandaGPT          18.556610   9.704767  6.499591  3.115937  
OLA               50.606540  16.605934  6.603703  2.900275

In [20]:
print(no_static_image_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 15.68 & 24.09 & 29.12 & 37.62 & 38.72 & 46.88 & 17.53 & 25.24 & 64.94 & 55.93 & 66.75 & 4.15 & 5.05 & 0.83 \\
DeepAVFusion & 12.07 & 13.46 & 25.65 & 27.18 & 22.91 & 41.15 & 12.43 & 14.86 & 46.99 & 33.12 & 58.62 & 3.97 & 3.37 & 0.22 \\
Equi-AV & 13.64 & 13.57 & 23.91 & 31.56 & 22.79 & 38.70 & 15.09 & 14.96 & 54.48 & 32.92 & 55.10 & 7.63 & 6.34 & 1.49 \\
AV-Siam & 15.12 & 24.70 & 27.04 & 36.18 & 39.44 & 42.91 & 16.51 & 26.53 & 62.45 & 56.97 & 61.09 & 10.76 & 8.58 & 3.93 \\
Gemini 1.5 Flash & 2.44 & 18.40 & 19.56 & 14.94 & 41.52 & 46.20 & 17.29 & 31.74 & 32.66 & 51.40 & 61.88 & 14.25 & 3.98 & 1.47 \\
Gemini 1.5 Pro & 3.63 & 27.02 & 26.64 & 20.12 & 54.78 & 57.09 & 19.93 & 33.15 & 35.61 & 73.45 & 78.62 & 3.04 & 3.95 & 0.58 \\
Gemi

# Table 12
**Audio-visual video classification results on the subset of VGGSounder that is labelled as containing voice over narrations**

In [21]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
# voice over
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(voice_over=True)
voice_over_ids = [video.video_id for video in vggsounder]


voice_over_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=voice_over_ids,
    dataset_path=csv_path
)
voice_over_table

  0%|          | 0/11 [00:00<?, ?it/s]

metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE            2.585628  21.690384  26.636364  28.791130  35.335473   
DeepAVFusion       1.567944  13.909942  21.371536  18.070841  23.062382   
Equi-AV            3.122901  11.772316  19.590909  23.694564  20.418768   
AV-Siam            2.451310  22.207848  23.363636  26.895565  35.824545   
Gemini 1.5 Flash   7.756884  16.041397  17.181818  33.693861  37.923963   
Gemini 1.5 Pro     9.973136  23.975852  15.681818  38.193941  50.950465   
Gemini 2.0 Flash   0.335796  14.877102  13.181818  11.210461  36.151787   
VideoLLaMA 2       5.473472  22.811557  26.681818  33.955485  49.326599   
Unified-IO 2       6.615178  14.316516  26.454545  33.023819  30.351121   
PandaGPT           4.130289   5.433376   6.272727  23.301594  20.174195   
OLA               20.047011   9.788702  13.045455  58.427918  25.754060   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           43.143045  12.638323  29.173420  54.063130  49.849073   
DeepAVFusion      34.888438   8.216282  18.090452  34.041812  32.634864   
Equi-AV           32.545932  12.085032  18.152350  44.492948  28.805520   
AV-Siam           37.696850  11.298777  30.145867  50.503694  50.539025   
Gemini 1.5 Flash  41.744361  36.973698  35.435435  70.080591  45.579991   
Gemini 1.5 Pro    51.055947  38.571429  34.712951  73.472129  67.442863   
Gemini 2.0 Flash  36.492357   5.012937  30.955994  18.334453  44.760673   
VideoLLaMA 2      52.506964  21.251919  42.666667  50.873069  51.746442   
Unified-IO 2      46.791045  28.976978  24.766355  53.156481  30.271669   
PandaGPT          21.406861  24.022663  18.223235  29.247817  17.766279   
OLA               45.651187  57.409133  24.016563  80.758899  24.579560   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           59.772727   4.757119  5.159129  0.737018  
DeepAVFusion      48.473462   3.892944  3.441084  0.139034  
Equi-AV           45.090909   9.246231  6.365159  1.373534  
AV-Siam           52.227273  11.591290  8.241206  3.785595  
Gemini 1.5 Flash  55.727273  34.103853  4.254606  3.182580  
Gemini 1.5 Pro    76.727273   6.733668  2.546064  1.373534  
Gemini 2.0 Flash  46.772727   2.948074  4.656616  1.139028  
VideoLLaMA 2      57.136364  16.180905  5.762144  3.015075  
Unified-IO 2      56.000000  17.453936  4.254606  1.608040  
PandaGPT          17.681818  15.711893  8.174204  4.656616  
OLA               55.136364  17.487437  5.058626  3.986600

In [22]:
print(voice_over_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 2.59 & 21.69 & 26.64 & 28.79 & 35.34 & 43.14 & 12.64 & 29.17 & 54.06 & 49.85 & 59.77 & 4.76 & 5.16 & 0.74 \\
DeepAVFusion & 1.57 & 13.91 & 21.37 & 18.07 & 23.06 & 34.89 & 8.22 & 18.09 & 34.04 & 32.63 & 48.47 & 3.89 & 3.44 & 0.14 \\
Equi-AV & 3.12 & 11.77 & 19.59 & 23.69 & 20.42 & 32.55 & 12.09 & 18.15 & 44.49 & 28.81 & 45.09 & 9.25 & 6.37 & 1.37 \\
AV-Siam & 2.45 & 22.21 & 23.36 & 26.90 & 35.82 & 37.70 & 11.30 & 30.15 & 50.50 & 50.54 & 52.23 & 11.59 & 8.24 & 3.79 \\
Gemini 1.5 Flash & 7.76 & 16.04 & 17.18 & 33.69 & 37.92 & 41.74 & 36.97 & 35.44 & 70.08 & 45.58 & 55.73 & 34.10 & 4.25 & 3.18 \\
Gemini 1.5 Pro & 9.97 & 23.98 & 15.68 & 38.19 & 50.95 & 51.06 & 38.57 & 34.71 & 73.47 & 67.44 & 76.73 & 6.73 & 2.55 & 1.37 \\
Gemini 2.

# Table 13 
**Audio-visual video classification results on the subset of VGGSounder that is labelled as not containing voice over narrations**

In [23]:

from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
# no voice over
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(voice_over=False)
no_voice_over_ids = [video.video_id for video in vggsounder]


no_voice_over_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=no_voice_over_ids,
    dataset_path=csv_path
)
no_voice_over_table


  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE           20.235412  24.816577  29.975505  40.718383  39.199201   
DeepAVFusion      15.542420  13.481986  26.787042  30.163900  22.576942   
Equi-AV           17.438851  13.831053  25.025515  34.411242  22.907550   
AV-Siam           19.534185  25.213167  28.056746  39.355760  39.709176   
Gemini 1.5 Flash   1.068537  19.016458  20.310267   9.964257  42.188620   
Gemini 1.5 Pro     2.020202  28.098354  29.475403  14.460052  55.573215   
Gemini 2.0 Flash   2.646298  16.289907  15.258216  13.086533  38.683330   
VideoLLaMA 2      17.939728  24.003569  27.770974  41.578047  50.036087   
Unified-IO 2      15.894482  15.169542  29.995918  36.477513  31.180577   
PandaGPT           2.996911   5.572080   6.593182  16.032077  19.442372   
OLA               13.774105  11.064842  21.096142  42.935729  27.323768   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           47.433132  25.085174  23.722842  67.852074  56.395003   
DeepAVFusion      42.228541  18.237839  13.333333  50.312935  32.486838   
Equi-AV           39.782859  21.034700  13.270699  57.342015  32.956573   
AV-Siam           43.830889  23.873817  24.897240  65.581434  57.128693   
Gemini 1.5 Flash  47.007981   8.387249  30.208891  21.930044  52.191156   
Gemini 1.5 Pro    58.496809  10.498789  32.851143  24.826780  74.261352   
Gemini 2.0 Flash  40.491471   9.962466  26.363636  19.091744  46.341463   
VideoLLaMA 2      53.526540  27.025053  37.943696  58.352116  52.349792   
Unified-IO 2      50.339596  24.006187  22.129086  50.914100  32.421178   
PandaGPT          21.521550  11.597008  19.152542  16.595709  16.904620   
OLA               46.673740  34.161363  17.497956  49.436514  25.332144   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           67.330067   4.203906  4.783184  0.810990  
DeepAVFusion      59.936272   4.057896  3.265271  0.249849  
Equi-AV           56.470708   7.414763  5.966567  1.415094  
AV-Siam           62.216779  11.345581  8.242304  3.814962  
Gemini 1.5 Flash  62.563789   9.111221  3.889441  0.951672  
Gemini 1.5 Pro    78.556848   2.391592  4.303211  0.422046  
Gemini 2.0 Flash  50.398040   2.706058  4.543198  0.984773  
VideoLLaMA 2      58.940600  15.640516  5.180404  3.053625  
Unified-IO 2      62.880180   8.854684  4.741807  1.787488  
PandaGPT          18.544601   7.952665  5.825885  2.548825  
OLA               49.050827  16.501159  6.554121  2.399868

In [24]:
print(no_voice_over_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 20.24 & 24.82 & 29.98 & 40.72 & 39.20 & 47.43 & 25.09 & 23.72 & 67.85 & 56.40 & 67.33 & 4.20 & 4.78 & 0.81 \\
DeepAVFusion & 15.54 & 13.48 & 26.79 & 30.16 & 22.58 & 42.23 & 18.24 & 13.33 & 50.31 & 32.49 & 59.94 & 4.06 & 3.27 & 0.25 \\
Equi-AV & 17.44 & 13.83 & 25.03 & 34.41 & 22.91 & 39.78 & 21.03 & 13.27 & 57.34 & 32.96 & 56.47 & 7.41 & 5.97 & 1.42 \\
AV-Siam & 19.53 & 25.21 & 28.06 & 39.36 & 39.71 & 43.83 & 23.87 & 24.90 & 65.58 & 57.13 & 62.22 & 11.35 & 8.24 & 3.81 \\
Gemini 1.5 Flash & 1.07 & 19.02 & 20.31 & 9.96 & 42.19 & 47.01 & 8.39 & 30.21 & 21.93 & 52.19 & 62.56 & 9.11 & 3.89 & 0.95 \\
Gemini 1.5 Pro & 2.02 & 28.10 & 29.48 & 14.46 & 55.57 & 58.50 & 10.50 & 32.85 & 24.83 & 74.26 & 78.56 & 2.39 & 4.30 & 0.42 \\
Gemini 

# Table 14
**Audio-visual video classification results on the subset of VGGSounder that is labelled as not containing background music, static images, or voice over narrations**

In [25]:

from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "cav-mae": "CAV-MAE",
    "deepavfusion": "DeepAVFusion",
    "equiav": "Equi-AV",
    "avsiam": "AV-Siam",
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}

# neither
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(
    static_image=False, 
    voice_over=False, 
    background_music=False
)
neither_ids = [video.video_id for video in vggsounder]

neither_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av", "a only", "v only"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    subset_ids=neither_ids,
    dataset_path=csv_path
)
neither_table

  0%|          | 0/11 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
CAV-MAE           20.327453  24.373031  29.631545  41.531379  39.687178   
DeepAVFusion      16.038984  13.662638  26.804957  31.481682  23.362436   
Equi-AV           17.640955  13.862634  25.054945  35.473710  23.530423   
AV-Siam           19.762481  25.305608  28.054299  40.333691  40.701272   
Gemini 1.5 Flash   1.049233  18.827977  20.361991   9.712022  42.778156   
Gemini 1.5 Pro     1.983166  27.410208  29.101487  14.276411  55.724334   
Gemini 2.0 Flash   2.663438  16.093258  15.475113  13.264763  39.063662   
VideoLLaMA 2      17.998386  23.755514  27.291532  41.945712  50.448934   
Unified-IO 2      16.095930  14.631380  29.786684  37.959216  30.592589   
PandaGPT           3.263000   5.217391   6.296057  17.149758  19.327731   
OLA               14.297244  10.838059  20.349063  44.789390  26.988323   

metric                                              hit_rate             \
modality                 av     a only     v only          a          v   
CAV-MAE           47.943123  21.522139  22.137983  69.168684  58.197858   
DeepAVFusion      43.260363  15.923999  13.312203  52.472627  34.274829   
Equi-AV           40.726167  18.231027  13.040182  59.079903  34.505356   
AV-Siam           44.598462  20.569449  22.592873  67.173988  59.684940   
Gemini 1.5 Flash  47.768207   8.971814  30.068966  21.457397  53.333333   
Gemini 1.5 Pro    58.797557  10.786298  31.654676  24.247665  74.883428   
Gemini 2.0 Flash  41.126061   8.637279  25.834543  19.024559  47.435413   
VideoLLaMA 2      53.788667  23.899371  37.031370  59.333564  53.686200   
Unified-IO 2      50.619157  22.098863  21.367521  53.545486  32.829238   
PandaGPT          21.560014  13.787902  19.848975  17.571774  17.114052   
OLA               46.345967  37.148640  16.544503  51.850571  25.356018   

metric                              mu                      
modality                 av          a         v        av  
CAV-MAE           69.308339   3.823328  5.101575  0.924446  
DeepAVFusion      62.553879   4.143844  3.381758  0.273875  
Equi-AV           58.875242   7.270030  6.334170  1.574983  
AV-Siam           64.473174  10.111847  8.730883  4.062999  
Gemini 1.5 Flash  63.865546   8.320018  4.028761  0.947272  
Gemini 1.5 Pro    78.965740   1.746177  4.439626  0.308149  
Gemini 2.0 Flash  51.182935   2.545081  4.873317  1.004337  
VideoLLaMA 2      60.413704  13.775394  5.489614  3.138553  
Unified-IO 2      64.964447   8.959142  4.896142  1.940196  
PandaGPT          19.056238   7.806437  5.900479  2.602146  
OLA               49.385908  17.222095  6.813513  2.716275

In [26]:
print(neither_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{5}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a only & v only & a & v & av & a & v & av \\
\midrule
CAV-MAE & 20.33 & 24.37 & 29.63 & 41.53 & 39.69 & 47.94 & 21.52 & 22.14 & 69.17 & 58.20 & 69.31 & 3.82 & 5.10 & 0.92 \\
DeepAVFusion & 16.04 & 13.66 & 26.80 & 31.48 & 23.36 & 43.26 & 15.92 & 13.31 & 52.47 & 34.27 & 62.55 & 4.14 & 3.38 & 0.27 \\
Equi-AV & 17.64 & 13.86 & 25.05 & 35.47 & 23.53 & 40.73 & 18.23 & 13.04 & 59.08 & 34.51 & 58.88 & 7.27 & 6.33 & 1.57 \\
AV-Siam & 19.76 & 25.31 & 28.05 & 40.33 & 40.70 & 44.60 & 20.57 & 22.59 & 67.17 & 59.68 & 64.47 & 10.11 & 8.73 & 4.06 \\
Gemini 1.5 Flash & 1.05 & 18.83 & 20.36 & 9.71 & 42.78 & 47.77 & 8.97 & 30.07 & 21.46 & 53.33 & 63.87 & 8.32 & 4.03 & 0.95 \\
Gemini 1.5 Pro & 1.98 & 27.41 & 29.10 & 14.28 & 55.72 & 58.80 & 10.79 & 31.65 & 24.25 & 74.88 & 78.97 & 1.75 & 4.44 & 0.31 \\
Gemini 

# Table 15 
**Audio-visual video classification results on VGGSound inputs.**

In [27]:
import pickle as pk
from vggsounder.benchmark import benchmark, get_metric_dataframe, select_metrics
from vggsounder.labels import VGGSounder


models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
vggsounder = VGGSounder(csv_path=csv_path)

vggsound_results = pk.load(open("precomputed_results/vggsound_results.pkl", "rb"))
table_15_df = get_metric_dataframe(vggsound_results, modalities=["a", "v", "av"])
table_15 = select_metrics(
    table_15_df, 
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ], 
    display_names=display_names
)
table_15
    

metric             accuracy                               f1             \
modality                  a          v         av          a          v   
Gemini 1.5 Flash   0.310760  22.122232  23.598343   1.708132  33.154648   
Gemini 1.5 Pro     1.288359  25.767189  21.306487   4.429014  36.409865   
Gemini 2.0 Flash   5.703742  20.290043  19.390133   9.947751  32.344071   
VideoLLaMA 2      27.981354  17.007640  21.461867  41.323123  31.461480   
Unified-IO 2      32.280202  20.238249  52.395442  43.714058  33.844973   
PandaGPT           5.198757   7.652467   8.947300  12.681473  16.831683   
OLA               10.714748   8.630066  14.294963  23.334031  17.811464   

metric                        hit_rate                               mu  \
modality                 av          a          v         av          a   
Gemini 1.5 Flash  35.938382   2.978117  31.833484  41.233976   1.514955   
Gemini 1.5 Pro    35.624527   6.111615  41.719539  45.701152   1.618542   
Gemini 2.0 Flash  33.913692   9.491130  30.551599  35.310113   2.499029   
VideoLLaMA 2      36.799180  30.046614  22.724330  27.897190  11.161466   
Unified-IO 2      64.064593  33.710993  22.840865  54.195261   4.875049   
PandaGPT          19.547942   8.539428  11.226207  13.297941   4.506021   
OLA               28.862026  18.056455  10.954292  22.413570   7.613622   

metric                                
modality                 v        av  
Gemini 1.5 Flash  4.169364  0.090638  
Gemini 1.5 Pro    5.412405  0.239544  
Gemini 2.0 Flash  4.771462  0.627994  
VideoLLaMA 2      2.848634  1.417843  
Unified-IO 2      3.418361  0.874013  
PandaGPT          4.480124  0.938754  
OLA               4.052829  0.712158

In [28]:
print(table_15.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{3}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a & v & av & a & v & av \\
\midrule
Gemini 1.5 Flash & 0.31 & 22.12 & 23.60 & 1.71 & 33.15 & 35.94 & 2.98 & 31.83 & 41.23 & 1.51 & 4.17 & 0.09 \\
Gemini 1.5 Pro & 1.29 & 25.77 & 21.31 & 4.43 & 36.41 & 35.62 & 6.11 & 41.72 & 45.70 & 1.62 & 5.41 & 0.24 \\
Gemini 2.0 Flash & 5.70 & 20.29 & 19.39 & 9.95 & 32.34 & 33.91 & 9.49 & 30.55 & 35.31 & 2.50 & 4.77 & 0.63 \\
VideoLLaMA 2 & 27.98 & 17.01 & 21.46 & 41.32 & 31.46 & 36.80 & 30.05 & 22.72 & 27.90 & 11.16 & 2.85 & 1.42 \\
Unified-IO 2 & 32.28 & 20.24 & 52.40 & 43.71 & 33.84 & 64.06 & 33.71 & 22.84 & 54.20 & 4.88 & 3.42 & 0.87 \\
PandaGPT & 5.20 & 7.65 & 8.95 & 12.68 & 16.83 & 19.55 & 8.54 & 11.23 & 13.30 & 4.51 & 4.48 & 0.94 \\
OLA & 10.71 & 8.63 & 14.29 & 23.33 & 17.81 & 28.86 & 18.06 & 10.95 & 22.41 & 7.61 & 4.05 & 0.71 \\
\bottomrule
\end{t

# Table 16 
**Audio-visual video classification results on VGGSound + human annotations**

In [29]:
from vggsounder.benchmark import benchmark


models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/intermediate-tables/0.4_majority_inhouse+mturk_formated.csv"
display_names = {
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
table_16 = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    dataset_path=csv_path
)
table_16

  0%|          | 0/7 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
Gemini 1.5 Flash   2.971230  25.187484  26.277176  11.860777  42.597151   
Gemini 1.5 Pro     4.290166  33.652271  30.043840  17.498566  53.350003   
Gemini 2.0 Flash   4.159722  22.291182  20.524291  12.695091  40.055483   
VideoLLaMA 2      29.857236  29.747436  35.626733  49.831245  52.755111   
Unified-IO 2      25.936662  20.455133  48.519281  45.403377  36.876975   
PandaGPT           5.130807   6.490820   7.649638  19.279765  19.434140   
OLA               19.928980  13.567796  22.528407  48.020778  29.272840   

metric                        hit_rate                               mu  \
modality                 av          a          v         av          a   
Gemini 1.5 Flash  46.365436  21.943619  45.565037  56.768364  11.615516   
Gemini 1.5 Pro    53.777022  25.182984  64.020343  69.240404   2.799091   
Gemini 2.0 Flash  41.127683  14.580767  41.953280  46.014136   2.479398   
VideoLLaMA 2      57.622272  52.300891  48.349280  55.748412  16.588519   
Unified-IO 2      63.909348  46.713530  30.342212  65.214279   9.477124   
PandaGPT          21.721358  14.885137  13.516076  15.424533   7.665530   
OLA               46.503629  45.698964  22.377381  43.741612  15.139244   

metric                                
modality                 v        av  
Gemini 1.5 Flash  4.156010  0.873828  
Gemini 1.5 Pro    4.887752  0.547030  
Gemini 2.0 Flash  4.937482  0.824098  
VideoLLaMA 2      5.363740  2.607275  
Unified-IO 2      4.695936  1.541631  
PandaGPT          5.314010  2.259165  
OLA               6.358340  2.152600

In [30]:
print(table_16.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{3}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a & v & av & a & v & av \\
\midrule
Gemini 1.5 Flash & 2.97 & 25.19 & 26.28 & 11.86 & 42.60 & 46.37 & 21.94 & 45.57 & 56.77 & 11.62 & 4.16 & 0.87 \\
Gemini 1.5 Pro & 4.29 & 33.65 & 30.04 & 17.50 & 53.35 & 53.78 & 25.18 & 64.02 & 69.24 & 2.80 & 4.89 & 0.55 \\
Gemini 2.0 Flash & 4.16 & 22.29 & 20.52 & 12.70 & 40.06 & 41.13 & 14.58 & 41.95 & 46.01 & 2.48 & 4.94 & 0.82 \\
VideoLLaMA 2 & 29.86 & 29.75 & 35.63 & 49.83 & 52.76 & 57.62 & 52.30 & 48.35 & 55.75 & 16.59 & 5.36 & 2.61 \\
Unified-IO 2 & 25.94 & 20.46 & 48.52 & 45.40 & 36.88 & 63.91 & 46.71 & 30.34 & 65.21 & 9.48 & 4.70 & 1.54 \\
PandaGPT & 5.13 & 6.49 & 7.65 & 19.28 & 19.43 & 21.72 & 14.89 & 13.52 & 15.42 & 7.67 & 5.31 & 2.26 \\
OLA & 19.93 & 13.57 & 22.53 & 48.02 & 29.27 & 46.50 & 45.70 & 22.38 & 43.74 & 15.14 & 6.36 & 2.15 \\
\bottomr

# Table 17
**Audio-visual video classification results on VGGSound + human annotations + automatically added labels**

In [31]:
from vggsounder.benchmark import benchmark


models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}
table_17 = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("accuracy", ["a", "v", "av"]),
        ("f1",["a", "v", "av"]), 
        ("hit_rate",["a", "v", "av"]), 
        ("mu",["a", "v", "av"])
    ],
    dataset_path=csv_path
)
table_17

  0%|          | 0/7 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


metric             accuracy                               f1             \
modality                  a          v         av          a          v   
Gemini 1.5 Flash   2.400214  18.460298  19.736623  14.485554  41.430015   
Gemini 1.5 Pro     3.603664  27.327690  26.946158  19.688358  54.751689   
Gemini 2.0 Flash   2.186267  16.025796  14.877480  12.694587  38.220739   
VideoLLaMA 2      15.457645  23.780734  27.571262  39.905384  49.905470   
Unified-IO 2      14.046935  15.010077  29.346558  35.706724  31.030268   
PandaGPT           3.222571   5.546151   6.534422  17.729805  19.579306   
OLA               15.023066  10.826280  19.619937  46.747671  27.029768   

metric                        hit_rate                               mu  \
modality                 av          a          v         av          a   
Gemini 1.5 Flash  46.095278  31.517015  50.955260  61.310218  14.061982   
Gemini 1.5 Pro    57.060436  34.512269  72.986699  78.221370   3.251709   
Gemini 2.0 Flash  39.788414  18.940964  46.045949  49.733289   2.753998   
VideoLLaMA 2      53.342149  56.863007  52.237001  58.609768  15.747561   
Unified-IO 2      49.716737  51.360567  32.019347  61.618603  10.558099   
PandaGPT          21.500710  19.114796  17.065699  18.386398   9.489681   
OLA               46.462882  55.672929  25.191455  50.166694  16.696529   

metric                                
modality                 v        av  
Gemini 1.5 Flash  3.961776  1.393589  
Gemini 1.5 Pro    3.955140  0.610525  
Gemini 2.0 Flash  4.565665  1.015329  
VideoLLaMA 2      5.295640  3.045988  
Unified-IO 2      4.645298  1.751941  
PandaGPT          6.291061  2.966355  
OLA               6.257880  2.714181

In [32]:
print(table_17.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrrrrrrrrrrr}
\toprule
metric & \multicolumn{3}{r}{accuracy} & \multicolumn{3}{r}{f1} & \multicolumn{3}{r}{hit_rate} & \multicolumn{3}{r}{mu} \\
modality & a & v & av & a & v & av & a & v & av & a & v & av \\
\midrule
Gemini 1.5 Flash & 2.40 & 18.46 & 19.74 & 14.49 & 41.43 & 46.10 & 31.52 & 50.96 & 61.31 & 14.06 & 3.96 & 1.39 \\
Gemini 1.5 Pro & 3.60 & 27.33 & 26.95 & 19.69 & 54.75 & 57.06 & 34.51 & 72.99 & 78.22 & 3.25 & 3.96 & 0.61 \\
Gemini 2.0 Flash & 2.19 & 16.03 & 14.88 & 12.69 & 38.22 & 39.79 & 18.94 & 46.05 & 49.73 & 2.75 & 4.57 & 1.02 \\
VideoLLaMA 2 & 15.46 & 23.78 & 27.57 & 39.91 & 49.91 & 53.34 & 56.86 & 52.24 & 58.61 & 15.75 & 5.30 & 3.05 \\
Unified-IO 2 & 14.05 & 15.01 & 29.35 & 35.71 & 31.03 & 49.72 & 51.36 & 32.02 & 61.62 & 10.56 & 4.65 & 1.75 \\
PandaGPT & 3.22 & 5.55 & 6.53 & 17.73 & 19.58 & 21.50 & 19.11 & 17.07 & 18.39 & 9.49 & 6.29 & 2.97 \\
OLA & 15.02 & 10.83 & 19.62 & 46.75 & 27.03 & 46.46 & 55.67 & 25.19 & 50.17 & 16.70 & 6.26 & 2.71 \\
\bottom

# Poster Tables

## Voice over

In [33]:

from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path= "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}

# no voice over
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(voice_over=False)
no_voice_over_ids = [video.video_id for video in vggsounder]


no_voice_over_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("f1",["a", "v", "av"]),
    ],
    subset_ids=no_voice_over_ids,
    dataset_path=csv_path
)
no_voice_over_table

# voice over
vggsounder.set_meta_filters(voice_over=True)
voice_over_ids = [video.video_id for video in vggsounder]


voice_over_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("f1",["a", "v", "av"]),
    ],
    subset_ids=voice_over_ids,
    dataset_path=csv_path
)
voice_over_table


diff_voice_over_table =  no_voice_over_table - voice_over_table
diff_voice_over_table


  0%|          | 0/7 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


  0%|          | 0/7 [00:00<?, ?it/s]

metric                   f1                    
modality                  a         v        av
Gemini 1.5 Flash -23.729605  4.264657  5.263620
Gemini 1.5 Pro   -23.733890  4.622750  7.440863
Gemini 2.0 Flash   1.876072  2.531543  3.999114
VideoLLaMA 2       7.622562  0.709488  1.019577
Unified-IO 2       3.453694  0.829456  3.548551
PandaGPT          -7.269517 -0.731823  0.114690
OLA              -15.492189  1.569708  1.022552

In [34]:
print(diff_voice_over_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrr}
\toprule
metric & \multicolumn{3}{r}{f1} \\
modality & a & v & av \\
\midrule
Gemini 1.5 Flash & -23.73 & 4.26 & 5.26 \\
Gemini 1.5 Pro & -23.73 & 4.62 & 7.44 \\
Gemini 2.0 Flash & 1.88 & 2.53 & 4.00 \\
VideoLLaMA 2 & 7.62 & 0.71 & 1.02 \\
Unified-IO 2 & 3.45 & 0.83 & 3.55 \\
PandaGPT & -7.27 & -0.73 & 0.11 \\
OLA & -15.49 & 1.57 & 1.02 \\
\bottomrule
\end{tabular}



## Background music

In [35]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path = "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}

# no background music
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(background_music=False)
no_background_music_ids = [video.video_id for video in vggsounder]

no_background_music_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("f1", ["a", "v", "av"]),
    ],
    subset_ids=no_background_music_ids,
    dataset_path=csv_path
)
no_background_music_table

# with background music
vggsounder = VGGSounder(background_music=True) 
background_music_ids = [video.video_id for video in vggsounder]

background_music_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("f1", ["a", "v", "av"]),
    ],
    subset_ids=background_music_ids,
    dataset_path=csv_path
)
background_music_table

diff_background_music_table = no_background_music_table - background_music_table
diff_background_music_table


  0%|          | 0/7 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


  0%|          | 0/7 [00:00<?, ?it/s]

metric                  f1                    
modality                 a         v        av
Gemini 1.5 Flash -1.377227  3.453764  4.351181
Gemini 1.5 Pro   -1.500017  0.853707  0.815063
Gemini 2.0 Flash  0.783692  2.703378  4.194068
VideoLLaMA 2      4.165877  2.344206  2.038672
Unified-IO 2      9.881900 -3.117212  2.421686
PandaGPT          4.126162 -1.177635  0.305579
OLA               8.579443 -3.099229 -2.594631

In [36]:
print(diff_background_music_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrr}
\toprule
metric & \multicolumn{3}{r}{f1} \\
modality & a & v & av \\
\midrule
Gemini 1.5 Flash & -1.38 & 3.45 & 4.35 \\
Gemini 1.5 Pro & -1.50 & 0.85 & 0.82 \\
Gemini 2.0 Flash & 0.78 & 2.70 & 4.19 \\
VideoLLaMA 2 & 4.17 & 2.34 & 2.04 \\
Unified-IO 2 & 9.88 & -3.12 & 2.42 \\
PandaGPT & 4.13 & -1.18 & 0.31 \\
OLA & 8.58 & -3.10 & -2.59 \\
\bottomrule
\end{tabular}



## Static Images

In [37]:
from vggsounder.benchmark import benchmark
from vggsounder.labels import VGGSounder

models_path = "../../supplimentary/models-results"
csv_path = "../../supplimentary/data/vggsounder+background-music.csv"
display_names = {
    "gemini-1.5-flash": "Gemini 1.5 Flash",
    "gemini-1.5-pro": "Gemini 1.5 Pro",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "video-llama-2-av": "VideoLLaMA 2",
    "unified-io-2": "Unified-IO 2",
    "pandagpt": "PandaGPT",
    "ola": "OLA",
}

# static images only
vggsounder = VGGSounder(csv_path=csv_path)
vggsounder.set_meta_filters(static_image=True)
static_image_ids = [video.video_id for video in vggsounder]

static_image_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("f1", ["a", "v", "av"]),
    ],
    subset_ids=static_image_ids,
    dataset_path=csv_path
)
static_image_table

# no static images
vggsounder.set_meta_filters(static_image=False)
no_static_image_ids = [video.video_id for video in vggsounder]

no_static_image_table = benchmark(
    models_path=models_path,
    display_names=display_names,
    metrics=[
        ("f1", ["a", "v", "av"]),
    ],
    subset_ids=no_static_image_ids,
    dataset_path=csv_path
)
no_static_image_table

diff_static_image_table = no_static_image_table - static_image_table
diff_static_image_table



  0%|          | 0/7 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['wind rustiling leaves'] will be ignored
  warnings.warn(


  0%|          | 0/7 [00:00<?, ?it/s]

/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(
/storage/slurm/zverev/repos/VGGSounder/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['female speech, woman someone'] will be ignored
  warnings.warn(


metric                  f1                    
modality                 a         v        av
Gemini 1.5 Flash  4.471900  2.175175  2.618375
Gemini 1.5 Pro    4.476055  0.745318  0.590651
Gemini 2.0 Flash -1.346644  2.665566  1.860792
VideoLLaMA 2     -4.610964  3.015794  0.487418
Unified-IO 2     -1.633012  3.409818  0.828950
PandaGPT          6.331558  3.022335  1.245494
OLA               6.841991  7.343210  3.062643

In [38]:
print(diff_static_image_table.to_latex(float_format="%.2f".__mod__))

\begin{tabular}{lrrr}
\toprule
metric & \multicolumn{3}{r}{f1} \\
modality & a & v & av \\
\midrule
Gemini 1.5 Flash & 4.47 & 2.18 & 2.62 \\
Gemini 1.5 Pro & 4.48 & 0.75 & 0.59 \\
Gemini 2.0 Flash & -1.35 & 2.67 & 1.86 \\
VideoLLaMA 2 & -4.61 & 3.02 & 0.49 \\
Unified-IO 2 & -1.63 & 3.41 & 0.83 \\
PandaGPT & 6.33 & 3.02 & 1.25 \\
OLA & 6.84 & 7.34 & 3.06 \\
\bottomrule
\end{tabular}

